## Import Libraries

In [33]:
import pandas as pd
import numpy as np
import sqlite3

from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from scipy.stats import zscore
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import sweetviz as sv


## 1. Data Understanding & Loading

**LOAD USERS DATA**

In [4]:
users = pd.read_csv('users.csv')

**LOAD SALES JSON DATA**

In [5]:
sales = pd.read_json('sales.json')


**LOAD INVENTORY SQL DATA**

In [6]:

# Create SQLite database in memory
conn = sqlite3.connect(":memory:")

# Read SQL file
with open("inventory.sql", "r", encoding="utf-8") as file:
    sql_script = file.read()

# Execute SQL script
conn.executescript(sql_script)

print("SQL File Loaded Successfully")

# Load products table
inventory = pd.read_sql_query(
    "SELECT * FROM products",
    conn
)



SQL File Loaded Successfully


In [7]:
#Merge Sales + Users
df = pd.merge(
    sales,
    users,
    on="user_id",
    how="left"
)

#Merge with Inventory
df = pd.merge(
    df,
    inventory,
    on="product_id",
    how="left"
)


**Display Top 5 Records**

In [8]:
print("============================== Data - Top 5 ===============================")
print(df.head())



============================== Data - Top 5 ===============================
  transaction_id user_id product_id  amount payment_type       date  \
0        T000001   U0024       P015   67.67       Wallet 2023-02-12   
1        T000002   U0196       P044   76.44          UPI 2023-03-24   
2        T000003   U0196       P049  104.57   Debit Card 2025-08-21   
3        T000004   U0133       P042  102.75  Net Banking 2024-07-23   
4        T000005   U0047       P038   23.89  Net Banking 2025-10-04   

             name  age  gender     city registration_date product_name  \
0   Aarohi Kapoor   20  Female    Patna        2023-03-15  Product_015   
1  Kabir Kulkarni   35   Other    Patna        2024-08-01  Product_044   
2  Kabir Kulkarni   35   Other    Patna        2024-08-01  Product_049   
3      Diya Singh   23  Female   Indore        2023-07-06  Product_042   
4     Anaya Gupta   28   Other  Kolkata        2022-10-13  Product_038   

  category    price  stock  
0    Books  2286.25    

**Identify Data Types**

In [9]:
print("============================== DATA TYPES ==============================")
print(df.dtypes)



============================== DATA TYPES ==============================
transaction_id                  str
user_id                         str
product_id                      str
amount                      float64
payment_type                    str
date                 datetime64[us]
name                            str
age                           int64
gender                          str
city                            str
registration_date               str
product_name                    str
category                        str
price                       float64
stock                         int64
dtype: object


**Check Missing Values**

In [10]:
print("============================== MISSING VALUES - Data ==============================")
print(df.isnull().sum())


============================== MISSING VALUES - Data ==============================
transaction_id       0
user_id              0
product_id           0
amount               0
payment_type         0
date                 0
name                 0
age                  0
gender               0
city                 0
registration_date    0
product_name         0
category             0
price                0
stock                0
dtype: int64


## 2. Data Cleaning

**SimpleImputer — Numerical Data**

In [11]:

numeric_cols = ['age', 'amount', 'price', 'stock']

mean_imputer = SimpleImputer(strategy='mean')

df[numeric_cols] = mean_imputer.fit_transform(
    df[numeric_cols]
)

**Categorical Missing Values**

In [12]:
categorical_cols = [
    'gender',
    'city',
    'category',
    'payment_type'
]

mode_imputer = SimpleImputer(strategy='most_frequent')

df[categorical_cols] = mode_imputer.fit_transform(
    df[categorical_cols]
)

**KNN Imputation**

In [13]:
knn = KNNImputer(n_neighbors=5)

df[numeric_cols] = knn.fit_transform(
    df[numeric_cols]
)

**Check Invalid/Inconsistent Values**

In [14]:
print("Invalid Age:")
print(df[(df['age'] < 0) | (df['age'] > 120)])

print("\nInvalid Amount:")
print(df[df['amount'] <= 0])

print("\nInvalid Price:")
print(df[df['price'] < 0])

print("\nInvalid Stock:")
print(df[df['stock'] < 0])

print("\nGender Values:")
print(df['gender'].unique())

print("\nCategory Values:")
print(df['category'].unique())

print("\nPayment Type Values:")
print(df['payment_type'].unique())

Invalid Age:
Empty DataFrame
Columns: [transaction_id, user_id, product_id, amount, payment_type, date, name, age, gender, city, registration_date, product_name, category, price, stock]
Index: []

Invalid Amount:
Empty DataFrame
Columns: [transaction_id, user_id, product_id, amount, payment_type, date, name, age, gender, city, registration_date, product_name, category, price, stock]
Index: []

Invalid Price:
Empty DataFrame
Columns: [transaction_id, user_id, product_id, amount, payment_type, date, name, age, gender, city, registration_date, product_name, category, price, stock]
Index: []

Invalid Stock:
Empty DataFrame
Columns: [transaction_id, user_id, product_id, amount, payment_type, date, name, age, gender, city, registration_date, product_name, category, price, stock]
Index: []

Gender Values:
<ArrowStringArray>
['Female', 'Other', 'Male']
Length: 3, dtype: str

Category Values:
<ArrowStringArray>
[      'Books',      'Beauty',        'Home',     'Grocery', 'Electronics',
    'Clo

## 3.OUTER HANDLING 

**1. Z-Score Outlier Detection & Removal**

In [15]:

numeric_cols = ['age', 'amount', 'price', 'stock']

z_scores = np.abs(zscore(df[numeric_cols]))

outliers = df[(z_scores > 3).any(axis=1)]

print("Number of Outliers:", len(outliers))
print(outliers)

df_zscore = df[(z_scores <= 3).all(axis=1)]

print("\nOriginal Rows:", len(df))
print("After Removing Outliers:", len(df_zscore))
print("Removed Rows:", len(df) - len(df_zscore))

Number of Outliers: 22
    transaction_id user_id product_id  amount payment_type       date  \
9          T000010   U0117       P006  550.95          UPI 2024-05-31   
20         T000021   U0031       P042  218.94   Debit Card 2025-10-17   
62         T000063   U0180       P019    7.81  Credit Card 2025-03-21   
278        T000279   U0055       P040  346.31          UPI 2025-10-02   
383        T000384   U0184       P012  213.24  Credit Card 2024-11-21   
414        T000415   U0018       P031  236.86  Credit Card 2024-08-10   
454        T000455   U0199       P019  255.70  Net Banking 2025-08-12   
555        T000556   U0134       P031  264.92          UPI 2025-09-02   
562        T000563   U0119       P015  253.67  Net Banking 2024-01-13   
622        T000623   U0177       P009  216.88       Wallet 2024-09-10   
647        T000648   U0192       P026  238.22          UPI 2023-04-22   
667        T000668   U0180       P001   66.73   Debit Card 2025-10-13   
680        T000681   U0046  

**IQR Outlier Detection & Removal**

In [16]:
numeric_cols = ['age', 'amount', 'price', 'stock']

Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = (
    (df[numeric_cols] < lower) |
    (df[numeric_cols] > upper)
)

print("Number of Outliers:")
print(outliers.sum())

df_iqr = df[~outliers.any(axis=1)]

print("\nOriginal Rows:", len(df))
print("After Removing Outliers:", len(df_iqr))
print("Removed Rows:", len(df) - len(df_iqr))

Number of Outliers:
age       20
amount    53
price      0
stock      0
dtype: int64

Original Rows: 1000
After Removing Outliers: 928
Removed Rows: 72


**3. Winsorization**

In [17]:
numeric_cols = ['age', 'amount', 'price', 'stock']

df_winsorized = df.copy()

for col in numeric_cols:
    lower = df[col].quantile(0.05)
    upper = df[col].quantile(0.95)

    df_winsorized[col] = df[col].clip(
        lower=lower,
        upper=upper
    )

print("Winsorization Completed")
print(df_winsorized[numeric_cols].head())

Winsorization Completed
    age  amount    price  stock
0  20.0   67.67  2286.25  251.0
1  35.0   76.44  1655.45  268.0
2  35.0  104.57  1701.17  159.0
3  23.0  102.75  4406.39   59.0
4  28.0   23.89   393.65   58.0


## 4. Data Transformation

**Convert date column**

In [18]:
df['date'] = pd.to_datetime(df['date'])

## Create separate date features
df['day'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['year']  = df['date'].dt.year

print(df[['date','month','year']].head())

        date  month  year
0 2023-02-12      2  2023
1 2023-03-24      3  2023
2 2025-08-21      8  2025
3 2024-07-23      7  2024
4 2025-10-04     10  2025


**Label Encoding — Binary Column**

In [19]:
le = LabelEncoder()
df['gender_encoded'] = le.fit_transform(df['gender'])
print(df[['gender', 'gender_encoded']].head())

   gender  gender_encoded
0  Female               0
1   Other               2
2   Other               2
3  Female               0
4   Other               2


**One-Hot Encoding — Nominal Columns**

In [20]:
df = pd.get_dummies(
    df,
    columns=['gender', 'city'],
    dtype = int
)

print(df.head())

  transaction_id user_id product_id  amount payment_type       date  \
0        T000001   U0024       P015   67.67       Wallet 2023-02-12   
1        T000002   U0196       P044   76.44          UPI 2023-03-24   
2        T000003   U0196       P049  104.57   Debit Card 2025-08-21   
3        T000004   U0133       P042  102.75  Net Banking 2024-07-23   
4        T000005   U0047       P038   23.89  Net Banking 2025-10-04   

             name   age registration_date product_name  ... city_Kolkata  \
0   Aarohi Kapoor  20.0        2023-03-15  Product_015  ...            0   
1  Kabir Kulkarni  35.0        2024-08-01  Product_044  ...            0   
2  Kabir Kulkarni  35.0        2024-08-01  Product_049  ...            0   
3      Diya Singh  23.0        2023-07-06  Product_042  ...            0   
4     Anaya Gupta  28.0        2022-10-13  Product_038  ...            1   

   city_Lucknow  city_Mumbai  city_Nagpur  city_Patna  city_Pune  city_Surat  \
0             0            0        

**Ordinal Encoding — Low, Medium, High**

In [21]:
df['spending_group'] = pd.cut(
    df['amount'],
    bins=[-np.inf, 100, 300, np.inf],
    labels=['Low', 'Medium', 'High']
)

df['spending_encoded'] = df['spending_group'].map({
    'Low': 0,
    'Medium': 1,
    'High': 2
})

print(df[['amount', 'spending_group', 'spending_encoded']].head())

   amount spending_group spending_encoded
0   67.67            Low                0
1   76.44            Low                0
2  104.57         Medium                1
3  102.75         Medium                1
4   23.89            Low                0


**Log and Square Root Transformation**

In [22]:
# Log transformation
df['amount_log'] = np.log1p(df['amount'])

# Square root transformation
df['amount_sqrt'] = np.sqrt(df['amount'])

print(df[['amount', 'amount_log', 'amount_sqrt']].head())

   amount  amount_log  amount_sqrt
0   67.67    4.229312     8.226178
1   76.44    4.349503     8.742997
2  104.57    4.659374    10.225947
3  102.75    4.641984    10.136567
4   23.89    3.214466     4.887740


## 5. Feature Scaling

In [23]:
numeric_cols = ['age','amount','price','stock']

#standardscaler
standard_scaler = StandardScaler()

df_standard = df.copy()

df_standard[numeric_cols] = standard_scaler.fit_transform(
    df[numeric_cols]
)

#MInMaxScaler
minmax_scaler = MinMaxScaler()

df_minmax = df.copy()

df_minmax[numeric_cols] = minmax_scaler.fit_transform(
    df[numeric_cols]
)

print("Original:")
print(df[numeric_cols].describe())

print("StandardScaler:")
print(df_standard[numeric_cols].describe())

print("MinMaxScaler:")
print(df_minmax[numeric_cols].describe())

Original:
               age       amount        price        stock
count  1000.000000  1000.000000  1000.000000  1000.000000
mean     31.279000    67.599040  2440.259140   232.893000
std       7.211183    45.411375  1257.318565   141.376989
min      18.000000     7.810000   128.310000    18.000000
25%      26.000000    37.745000  1636.955000   134.000000
50%      31.000000    56.390000  2390.820000   229.000000
75%      35.000000    82.935000  3416.650000   332.000000
max      53.000000   550.950000  4970.990000   496.000000
StandardScaler:
                age        amount         price        stock
count  1.000000e+03  1.000000e+03  1.000000e+03  1000.000000
mean   1.243450e-17 -3.730349e-17 -8.704149e-17     0.000000
std    1.000500e+00  1.000500e+00  1.000500e+00     1.000500
min   -1.842367e+00 -1.317268e+00 -1.839714e+00    -1.520760
25%   -7.324237e-01 -6.577422e-01 -6.392223e-01    -0.699849
50%   -3.870927e-02 -2.469568e-01 -3.934077e-02    -0.027550
75%    5.162623e-01  3.37

## 6. Feature Construction

**Average Monthly Spend**

In [24]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

monthly_spend = df.groupby(
    ['user_id', df['date'].dt.to_period('M')]
)['amount'].sum()

avg_monthly_spend = monthly_spend.groupby(
    level=0
).mean()

df['avg_monthly_spend'] = df['user_id'].map(
    avg_monthly_spend
)

print("Average Monthly Spend per Customer:")
print(df[['user_id', 'avg_monthly_spend']].drop_duplicates().head(10))

Average Monthly Spend per Customer:
   user_id  avg_monthly_spend
0    U0024          75.430000
1    U0196          88.997143
3    U0133          90.230000
4    U0047          40.737500
6    U0086          87.285000
7    U0042          49.590000
8    U0074          60.693333
9    U0117         148.548889
10   U0083          59.123750
11   U0169          71.077500


**Frequency of Purchase**

In [25]:
purchase_frequency = df.groupby(
    'user_id'
)['transaction_id'].count()

df['purchase_frequency'] = df['user_id'].map(
    purchase_frequency
)

print("Frequency of Purchase:")
print(df[['user_id', 'purchase_frequency']].drop_duplicates().head(10))

Frequency of Purchase:
   user_id  purchase_frequency
0    U0024                  11
1    U0196                   7
3    U0133                   5
4    U0047                   4
6    U0086                   4
7    U0042                   3
8    U0074                   6
9    U0117                  13
10   U0083                   8
11   U0169                   4


**Days Since Last Purchase**

In [26]:
last_purchase = df.groupby(
    'user_id'
)['date'].max()

today = df['date'].max()

days_since_last = (today - last_purchase).dt.days

df['days_since_last_purchase'] = df['user_id'].map(
    days_since_last
)

print("Days Since Last Purchase:")
print(df[['user_id', 'days_since_last_purchase']].drop_duplicates().head(10))

Days Since Last Purchase:
   user_id  days_since_last_purchase
0    U0024                        90
1    U0196                        72
3    U0133                       370
4    U0047                        28
6    U0086                        24
7    U0042                        21
8    U0074                       258
9    U0117                        56
10   U0083                         9
11   U0169                       169


**Category-wise Total Expenditure**

In [27]:
category_total = df.groupby(
    'category'
)['amount'].sum()

df['category_total_expenditure'] = df['category'].map(
    category_total
)

print("Category-wise Total Expenditure:")
print(category_total)

Category-wise Total Expenditure:
category
Beauty          8079.73
Books          12529.84
Clothing        9080.87
Electronics     5616.05
Grocery         8427.07
Home           13243.42
Sports          4606.07
Toys            6015.99
Name: amount, dtype: float64


## Q7. Final Dataset Preparation

**Merge all cleaned and engineered data**

In [28]:
# Final dataset

final_df = df.copy()

print("Final Dataset:")
print(final_df.head())

print("Final Shape:")
print(final_df.shape)

Final Dataset:
  transaction_id user_id product_id  amount payment_type       date  \
0        T000001   U0024       P015   67.67       Wallet 2023-02-12   
1        T000002   U0196       P044   76.44          UPI 2023-03-24   
2        T000003   U0196       P049  104.57   Debit Card 2025-08-21   
3        T000004   U0133       P042  102.75  Net Banking 2024-07-23   
4        T000005   U0047       P038   23.89  Net Banking 2025-10-04   

             name   age registration_date product_name  ... city_Vadodara  \
0   Aarohi Kapoor  20.0        2023-03-15  Product_015  ...             0   
1  Kabir Kulkarni  35.0        2024-08-01  Product_044  ...             0   
2  Kabir Kulkarni  35.0        2024-08-01  Product_049  ...             0   
3      Diya Singh  23.0        2023-07-06  Product_042  ...             0   
4     Anaya Gupta  28.0        2022-10-13  Product_038  ...             0   

   city_Visakhapatnam  spending_group  spending_encoded  amount_log  \
0                   0   

**Number of records before and after cleaning**

In [29]:
print("Number of Records Before Cleaning:", len(df))
print("Number of Record After Cleaning:", len(final_df))


Number of Records Before Cleaning: 1000
Number of Record After Cleaning: 1000


**Number of features created**

In [30]:
original_features = [
    'user_id',
    'transaction_id',
    'product_id',
    'age',
    'gender',
    'city',
    'amount',
    'payment_type',
    'date',
    'product_name',
    'category',
    'price',
    'stock'
]

created_features = [
    col for col in final_df.columns
    if col not in original_features
]

print("Number of Features Created:", len(created_features))
print("Features Created:")
print(created_features)

Number of Features Created: 37
Features Created:
['name', 'registration_date', 'day', 'month', 'year', 'gender_encoded', 'gender_Female', 'gender_Male', 'gender_Other', 'city_Ahmedabad', 'city_Bengaluru', 'city_Bhopal', 'city_Chennai', 'city_Delhi', 'city_Ghaziabad', 'city_Hyderabad', 'city_Indore', 'city_Jaipur', 'city_Kanpur', 'city_Kolkata', 'city_Lucknow', 'city_Mumbai', 'city_Nagpur', 'city_Patna', 'city_Pune', 'city_Surat', 'city_Thane', 'city_Vadodara', 'city_Visakhapatnam', 'spending_group', 'spending_encoded', 'amount_log', 'amount_sqrt', 'avg_monthly_spend', 'purchase_frequency', 'days_since_last_purchase', 'category_total_expenditure']


**Outlier count — Before vs After**

In [31]:

numeric_cols = ['age', 'amount', 'price', 'stock']

# Before
z_before = np.abs(zscore(df[numeric_cols]))
outliers_before = (z_before > 3).any(axis=1).sum()

# After
z_after = np.abs(zscore(final_df[numeric_cols]))
outliers_after = (z_after > 3).any(axis=1).sum()

print("Outliers Before Cleaning:", outliers_before)
print("Outliers After Cleaning:", outliers_after)

Outliers Before Cleaning: 22
Outliers After Cleaning: 22


**Q8. Bonus — Pandas Profiling / YData Profiling**

In [32]:
import sweetviz as sv

report = sv.analyze(df)

report.show_html("sweetviz_report.html")

                                             |          | [  0%]   00:00 -> (? left)

Report sweetviz_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.
